<a href="https://colab.research.google.com/github/mugishaalex24682-cloud/FRESH_SMART_QUEUE_ML.IPYNB/blob/main/smart_queue_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

print("=" * 60)
print("FILES IN COLAB ROOT")
print("=" * 60)
for f in sorted(os.listdir('.')):
    print(f)

FILES IN COLAB ROOT
.config
CheckBloodPressure.csv
CheckPatientType.csv
Fill_Information.csv
MedicalRecord1.csv
MedicalRecord2.csv
MedicalRecord3.csv
MedicalRecord4.csv
OutPatientDepartment.csv
OutPatientDepartment.txt
Triage.csv
Triage_ReadMe.txt
diagnosis.csv.gz
drive
edstays.csv.gz
medrecon.csv.gz
pyxis.csv.gz
sample_data
triage.csv.gz
vitalsign.csv.gz


In [ ]:
# Cell 1 — Load MIMIC-IV-ED data
import pandas as pd
import numpy as np

# Load
triage = pd.read_csv('triage.csv.gz')
edstays = pd.read_csv('edstays.csv.gz')

print("=" * 60)
print("RAW SHAPES")
print("=" * 60)
print(f"triage:  {triage.shape}")
print(f"edstays: {edstays.shape}")

# Merge on stay_id to bring in demographics
df = triage.merge(
    edstays[['stay_id', 'gender', 'race', 'arrival_transport']],
    on='stay_id',
    how='left'
)

print(f"\nMerged shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

print("\n" + "=" * 60)
print("FIRST 3 ROWS")
print("=" * 60)
print(df.head(3))

print("\n" + "=" * 60)
print("ACUITY DISTRIBUTION (target before binarising)")
print("=" * 60)
print(df['acuity'].value_counts().sort_index())

print("\n" + "=" * 60)
print("MISSING VALUES PER COLUMN")
print("=" * 60)
print(df.isna().sum().sort_values(ascending=False))

RAW SHAPES
triage:  (222, 11)
edstays: (222, 9)

Merged shape: (222, 14)

Columns: ['subject_id', 'stay_id', 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint', 'gender', 'race', 'arrival_transport']

FIRST 3 ROWS
   subject_id   stay_id  temperature  heartrate  resprate  o2sat  sbp  dbp  \
0    10016742  33211001          NaN        NaN       NaN    NaN  NaN  NaN   
1    10032725  30701739          NaN        NaN       NaN    NaN  NaN  NaN   
2    10010867  30115213          NaN        NaN       NaN    NaN  NaN  NaN   

  pain  acuity        chiefcomplaint gender                    race  \
0  NaN     NaN             PICC EVAL      F  BLACK/AFRICAN AMERICAN   
1  NaN     NaN          FACIAL DROOP      F  BLACK/AFRICAN AMERICAN   
2  NaN     NaN  MVC/INTUBATED TRAUMA      F       WHITE - BRAZILIAN   

  arrival_transport  
0         AMBULANCE  
1         AMBULANCE  
2         AMBULANCE  

ACUITY DISTRIBUTION (target before binarising)
acuit

In [ ]:
# Cell 2 — Binarise target and drop rows with missing acuity

# Drop rows where acuity is missing (15 rows)
df = df.dropna(subset=['acuity']).copy()

# Binarise: ESI 1 or 2 = urgent (1), ESI 3 or 4 = not urgent (0)
df['is_urgent'] = (df['acuity'] <= 2).astype(int)

print("=" * 60)
print("AFTER BINARISING")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['is_urgent'].value_counts().sort_index())
print(f"\nProportions:")
print(df['is_urgent'].value_counts(normalize=True).round(3))

print("\n" + "=" * 60)
print("SANITY CHECK — acuity vs binarised target")
print("=" * 60)
print(df.groupby('acuity')['is_urgent'].agg(['count', 'mean']))

AFTER BINARISING
Shape: (207, 15)

Target distribution:
is_urgent
0     92
1    115
Name: count, dtype: int64

Proportions:
is_urgent
1    0.556
0    0.444
Name: proportion, dtype: float64

SANITY CHECK — acuity vs binarised target
        count  mean
acuity             
1.0        18   1.0
2.0        97   1.0
3.0        90   0.0
4.0         2   0.0
